# **Prédire le nombre de jour avant une prochaine commande**

## Import

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, KFold
from scipy.stats import randint, uniform

## Modèle

In [2]:
df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")
df = df.dropna()

orders = df.groupby("order_id").agg(
    user_id                = ("user_id",               "first"),
    order_number           = ("order_number",           "first"),
    order_hour_of_day      = ("order_hour_of_day",      "first"),
    order_dow              = ("order_dow",              "first"),
    days_since_prior_order = ("days_since_prior_order", "first"),
).reset_index()

orders = orders.sort_values(["user_id", "order_number"]).reset_index(drop=True)

orders["is_weekend"] = orders["order_dow"].isin([0, 6]).astype(int)
orders["time_of_day"] = pd.cut(
    orders["order_hour_of_day"],
    bins=[0, 6, 12, 17, 21, 24],
    labels=["nuit", "matin", "apres_midi", "soir", "soiree"],
    right=False
).astype(str)
orders['user_order_count'] = orders.groupby('user_id').cumcount()
orders["user_avg_days"] = orders.groupby("user_id")["days_since_prior_order"].transform(
    lambda x: x.shift(1).expanding().mean()
)
orders["user_std_days"] = orders.groupby("user_id")["days_since_prior_order"].transform(
    lambda x: x.shift(1).expanding().std()
)

orders = orders.dropna()
next_days_since_order = orders['days_since_prior_order'].shift(-1)
tagets = ["is_weekend","time_of_day","order_hour_of_day","order_dow","user_order_count","user_avg_days","user_std_days"]

X2 = pd.get_dummies(orders[tagets])
y2 = orders.groupby('user_id')['days_since_prior_order'].shift(-1)

mask = y2.notna()
X2 = X2[mask]
y2 = y2[mask]

model = HistGradientBoostingRegressor(
    max_iter=661,                 
    learning_rate=np.float64(0.10563362070328196),
    max_leaf_nodes=161,            
    min_samples_leaf=10,                    
    random_state=4,
    l2_regularization= np.float64(0.36480308916903204),
    max_depth=5
)

X_train, X_test, y_train, y_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test , y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(mae)
print(rmse)
print(r2)

#/---------------------------------------------------------------\#
#|------------------------ Justification ------------------------|#
#\---------------------------------------------------------------/#

# param_dist = {
#     "max_iter": randint(200, 800),
#     "learning_rate": uniform(0.01, 0.15),
#     "max_leaf_nodes": randint(31, 512),
#     "min_samples_leaf": randint(10, 100),
#     "l2_regularization": uniform(0.0, 0.5),
#     "max_depth": randint(3, 12),
#     "random_state": randint(0, 200)
# }

# cv = KFold(n_splits=5, shuffle=True, random_state=42)

# search = RandomizedSearchCV(
#     estimator = HistGradientBoostingRegressor(),
#     param_distributions = param_dist,
#     n_iter = 30,
#     scoring = "r2",
#     cv = cv,
#     n_jobs = -1,
#     verbose = 2,
#     random_state= 42,
#     return_train_score = True
# )

# search.fit(X_train, y_train)

# print(search.best_params_)
# print(f"Meilleur R² (CV) : {search.best_score_:.4f}")

# best_model = search.best_estimator_
# y_pred = best_model.predict(X_test)
# r2_2 = r2_score(y_test, y_pred)
# print(f"R² final (test) : {r2_2}")


4.82513171457204
6.7256011478743245
0.21240208229414081
